# deBerta Trasformer solution
---





In [ ]:
import torch, psutil

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("RAM (GB):", round(psutil.virtual_memory().total / 1e9, 1))

GPU: NVIDIA A100-SXM4-80GB
VRAM (GB): 85.2
RAM (GB): 179.4


In [ ]:
import torch
import pandas as pd
import numpy as np

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

from transformers import (
	DebertaV2Tokenizer,
	DebertaV2ForSequenceClassification,
	Trainer,
	TrainingArguments,
	EarlyStoppingCallback
)


In [ ]:
DEV_PATH  = "/content/development.csv"
EVAL_PATH = "/content/evaluation.csv"

df_dev  = pd.read_csv(DEV_PATH)
df_eval = pd.read_csv(EVAL_PATH)

for df in (df_dev, df_eval):
	df["title"]   = df["title"].fillna("").astype(str)
	df["article"] = df["article"].fillna("").astype(str)


In [ ]:
def build_text(df):
	return df["title"] + "\n\n" + df["article"]

df_dev["text"]  = build_text(df_dev)
df_eval["text"] = build_text(df_eval)


In [ ]:
train_df, val_df = train_test_split(
	df_dev,
	test_size=0.15,
	stratify=df_dev["label"],
	random_state=42
)


In [ ]:
class NewsDataset(torch.utils.data.Dataset):
	def __init__(self, df, tokenizer):
		self.texts  = df["text"].tolist()
		self.labels = df["label"].values
		self.tokenizer = tokenizer

	def __len__(self):
		return len(self.labels)

	def __getitem__(self, idx):
		enc = self.tokenizer(
			self.texts[idx],
			truncation=True,
			padding="max_length",
			max_length=512,
			return_tensors="pt"
		)
		item = {k: v.squeeze(0) for k,v in enc.items()}
		item["labels"] = torch.tensor(self.labels[idx])
		return item


In [ ]:
MODEL_NAME = "microsoft/deberta-v3-large"

tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)

train_ds = NewsDataset(train_df, tokenizer)
val_ds   = NewsDataset(val_df, tokenizer)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
model = DebertaV2ForSequenceClassification.from_pretrained(
	MODEL_NAME,
	num_labels=7
).cuda()


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def compute_metrics(eval_pred):
	logits, labels = eval_pred
	preds = logits.argmax(axis=1)
	return {"macro_f1": f1_score(labels, preds, average="macro")}


In [ ]:
args = TrainingArguments(
	output_dir="./deberta_out",
	learning_rate=2e-5,
	per_device_train_batch_size=16,
	per_device_eval_batch_size=16,
	num_train_epochs=3,
	fp16=True,


	eval_strategy="epoch",
	save_strategy="epoch",

	load_best_model_at_end=True,
	metric_for_best_model="macro_f1",

	logging_steps=100,
	report_to="none",
	save_total_limit=2
)


In [ ]:
import transformers
print(transformers.__version__)

4.57.3


In [ ]:
from transformers import TrainingArguments
import inspect

print(TrainingArguments)
print(inspect.getfile(TrainingArguments))

<class 'transformers.training_args.TrainingArguments'>
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py


In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,0.686800,0.656868,0.721789
2,0.532900,0.622965,0.742744
3,0.397900,0.671432,0.744979


TrainOutput(global_step=12750, training_loss=0.597928825378418, metrics={'train_runtime': 5680.8392, 'train_samples_per_second': 35.909, 'train_steps_per_second': 2.244, 'total_flos': 1.9011010577134694e+17, 'train_loss': 0.597928825378418, 'epoch': 3.0})

In [ ]:
import random, numpy as np, torch

seed = 1337
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


In [ ]:
import random, numpy as np, torch

seed = 1337
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


In [ ]:
train_df, val_df = train_test_split(
	df_dev,
	test_size=0.15,
	stratify=df_dev["label"],
	random_state=1337
)


In [ ]:
class NewsDatasetEval(torch.utils.data.Dataset):
	def __init__(self, df, tokenizer):
		self.texts = df["text"].tolist()
		self.tokenizer = tokenizer

	def __len__(self):
		return len(self.texts)

	def __getitem__(self, idx):
		enc = self.tokenizer(
			self.texts[idx],
			truncation=True,
			padding="max_length",
			max_length=512,
			return_tensors="pt"
		)
		return {k: v.squeeze(0) for k,v in enc.items()}


In [ ]:
eval_ds = NewsDatasetEval(df_eval, tokenizer)


In [ ]:
pred_out = trainer.predict(eval_ds)
logits = pred_out.predictions


In [ ]:
print(logits.shape)

(20000, 7)


In [ ]:
import numpy as np
np.save("logits_seed42.npy", logits)


In [ ]:
preds = logits.argmax(axis=1)

submission = pd.DataFrame({
	"Id": df_eval["Id"].astype(int),
	"Predicted": preds.astype(int)
})

submission.to_csv("submission_seed42.csv", index=False)


In [ ]:
np.save("logits_seed1337.npy", logits)


In [ ]:
import numpy as np

logits_ens = (
	np.load("logits_seed42.npy") +
	np.load("logits_seed1337.npy")
) / 2

preds = logits_ens.argmax(axis=1)

submission = pd.DataFrame({
	"Id": df_eval["Id"].astype(int),
	"Predicted": preds.astype(int)
})

submission.to_csv("submission_ensemble.csv", index=False)
